# DeepEval — RAG Evaluation Practical

LangChain RAG + DeepEval 5 core RAG metrics:
- Answer Relevancy
- Faithfulness
- Contextual Relevancy
- Contextual Precision
- Contextual Recall


In [ ]:
%pip install -U deepeval langchain langchain-openai pandas

In [4]:
import os
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)


In [5]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

RAG_MODEL = os.getenv("RAG_MODEL", "gpt-4.1-mini")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
DEEPEVAL_JUDGE_MODEL = os.getenv("DEEPEVAL_JUDGE_MODEL", "gpt-4.1-mini")

print(RAG_MODEL, EMBEDDING_MODEL, DEEPEVAL_JUDGE_MODEL)


gpt-4.1-mini text-embedding-3-small gpt-4.1-mini


In [6]:
import os

os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "300"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "1"


from deepeval.config.settings import reset_settings

reset_settings(
    reload_dotenv=True
)

print("DeepEval timeout settings reloaded.")

DeepEval timeout settings reloaded.


In [7]:
documents = [
    Document(
        page_content="Full-time employees receive 24 paid leaves per calendar year.",
        metadata={"doc_id": "leave_policy"},
    ),
    Document(
        page_content="Employees are allowed to work from home for a maximum of 2 days per week.",
        metadata={"doc_id": "remote_policy"},
    ),
    Document(
        page_content="Employees can claim up to ₹3000 per month for internet reimbursement.",
        metadata={"doc_id": "internet_policy"},
    ),
    Document(
        page_content="The standard probation period for new employees is 6 months.",
        metadata={"doc_id": "probation_policy"},
    ),
    Document(
        page_content="Employees receive ₹1000 per month as mobile reimbursement.",
        metadata={"doc_id": "mobile_policy"},
    ),
    Document(
        page_content="Medical insurance coverage begins from the employee's date of joining.",
        metadata={"doc_id": "insurance_policy"},
    ),
]


In [ ]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(documents)

TOP_K = 3

retriever = vector_store.as_retriever(
    search_kwargs={"k": TOP_K}
)

llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)


In [9]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY from the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not present in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }


In [ ]:
sample = rag_pipeline(
    "What is the monthly internet reimbursement limit?"
)

print("ANSWER:")
print(sample["answer"])

print("\nRETRIEVED DOCS:")
print(sample["retrieved_doc_ids"])

print("\nCONTEXT:")
for chunk in sample["retrieval_context"]:
    print("-", chunk)


ANSWER:
The monthly internet reimbursement limit is ₹3000.

RETRIEVED DOCS:
['internet_policy', 'mobile_policy', 'remote_policy']

CONTEXT:
- Employees can claim up to ₹3000 per month for internet reimbursement.
- Employees receive ₹1000 per month as mobile reimbursement.
- Employees are allowed to work from home for a maximum of 2 days per week.


In [11]:
goldens = [
    Golden(
        input="How many paid leaves does a full-time employee receive?",
        expected_output="A full-time employee receives 24 paid leaves per calendar year.",
    ),
    Golden(
        input="How many work-from-home days are allowed per week?",
        expected_output="Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="What is the monthly internet reimbursement limit?",
        expected_output="Employees can claim up to ₹3000 per month for internet reimbursement.",
    ),
    Golden(
        input="What is the probation period for new employees?",
        expected_output="The standard probation period for new employees is 6 months.",
    ),
    Golden(
        input="Can an employee work remotely for 3 days every week?",
        expected_output="No. Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="When does employee medical insurance coverage begin?",
        expected_output="Medical insurance coverage begins from the employee's date of joining.",
    ),
]

dataset = EvaluationDataset(goldens=goldens)

print("Goldens:", len(dataset.goldens))


Goldens: 6


In [12]:
rag_runs = []

for i, golden in enumerate(dataset.goldens, start=1):
    print(f"Running RAG {i}/{len(dataset.goldens)}")

    result = rag_pipeline(golden.input)

    rag_runs.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "actual_output": result["answer"],
        "retrieval_context": result["retrieval_context"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
    })


Running RAG 1/6
Running RAG 2/6
Running RAG 3/6
Running RAG 4/6
Running RAG 5/6
Running RAG 6/6


In [13]:
pd.DataFrame([
    {
        "input": run["input"],
        "expected_output": run["expected_output"],
        "actual_output": run["actual_output"],
        "retrieved_doc_ids": run["retrieved_doc_ids"],
    }
    for run in rag_runs
])


,input,expected_output,actual_output,retrieved_doc_ids
0,How many paid leaves does a full-time employee...,A full-time employee receives 24 paid leaves p...,A full-time employee receives 24 paid leaves p...,"[leave_policy, remote_policy, mobile_policy]"
1,How many work-from-home days are allowed per w...,Employees can work from home for a maximum of ...,Employees are allowed to work from home for a ...,"[remote_policy, leave_policy, internet_policy]"
2,What is the monthly internet reimbursement limit?,Employees can claim up to ₹3000 per month for ...,The monthly internet reimbursement limit is ₹3...,"[internet_policy, mobile_policy, remote_policy]"
3,What is the probation period for new employees?,The standard probation period for new employee...,The probation period for new employees is 6 mo...,"[probation_policy, insurance_policy, remote_po..."
4,Can an employee work remotely for 3 days every...,No. Employees can work from home for a maximum...,"No, employees are allowed to work from home fo...","[remote_policy, internet_policy, leave_policy]"
5,When does employee medical insurance coverage ...,Medical insurance coverage begins from the emp...,Employee medical insurance coverage begins fro...,"[insurance_policy, leave_policy, internet_policy]"


In [14]:
test_cases = [
    LLMTestCase(
        input=run["input"],
        actual_output=run["actual_output"],
        expected_output=run["expected_output"],
        retrieval_context=run["retrieval_context"],
    )
    for run in rag_runs
]

print("Test cases:", len(test_cases))


Test cases: 6


In [15]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)


answer_relevancy = AnswerRelevancyMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

faithfulness = FaithfulnessMetric(
    threshold=0.85,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_recall = ContextualRecallMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)


rag_metrics = [
    answer_relevancy,
    faithfulness,
    contextual_relevancy,
    contextual_precision,
    contextual_recall,
]

In [16]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)


answer_relevancy = AnswerRelevancyMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

faithfulness = FaithfulnessMetric(
    threshold=0.85,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_recall = ContextualRecallMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)


rag_metrics = [
    answer_relevancy,
    faithfulness,
    contextual_relevancy,
    contextual_precision,
    contextual_recall,
]

In [17]:
test_case = test_cases[0]

for metric in rag_metrics:

    print("\n", "=" * 70)
    print(metric.__class__.__name__)

    metric.measure(test_case)

    print("Score:", metric.score)
    print("Passed:", metric.is_successful())
    print("Reason:", metric.reason)

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


AnswerRelevancyMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the response directly and fully addresses the question about paid leaves for full-time employees without any irrelevant information.

FaithfulnessMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!

ContextualRelevancyMetric


Score: 0.3333333333333333
Passed: False
Reason: The score is 0.33 because while the relevant statement 'Full-time employees receive 24 paid leaves per calendar year.' directly answers the question, the presence of unrelated statements about work-from-home days and mobile reimbursement dilutes the overall relevancy of the retrieval context.

ContextualPrecisionMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the first node in retrieval contexts clearly provides the exact number of paid leaves, stating 'Full-time employees receive 24 paid leaves per calendar year,' which directly answers the question. The subsequent nodes, ranked lower, discuss unrelated topics like working from home days and mobile reimbursement, appropriately placed below the relevant node.

ContextualRecallMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the sentence in the expected output exactly matches the information provided in node 1 of the retrieval context, with no conflicting or missing details.


In [18]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig


evaluation_result = evaluate(
    test_cases=test_cases[:1],
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

evaluation_result

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              How many paid leaves does a full-time employee receive?                              │
│  │     Actual Output:      A full-time employee receives 24 paid leaves per calendar year.                      │
│  │     Expected Output:    A full-time employee receives 24 paid leaves per calendar year.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.80      │ The score is 1.00 because the response directly...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.85      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 0.75      │ The score is 0.33 because while the relevant          │
│              │                      │       │           │ statement 'Full-time employees receive 24 paid        │
│              │                      │       │           │ leaves per calendar year.' directly answers the       │
│              │                      │       │           │ question, the presence of irrelevant statements       │
│              │                      │       │           │ about work-from-home days and mobile reimbursement    │
│              │                      │       │           │ lowers the overall contextual relevancy.              │
│        PASS  │ Contextual Precision │ 1.00  │ 0.75      │ The score is 1.00 because the first node in ret...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.80      │ The score is 1.00 because the sentence in the e...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Answer Relevancy               │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Faithfulness                   │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Contextual Relevancy           │ 0.33                 │ 0.00% | passed=0 | failed=1                 │ 1        │
│  Contextual Precision           │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Contextual Recall              │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
╰─────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=16639308;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 28.85s | token cost: 0.0036092 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.8, success=True, score=1.0, reason='The score is 1.00 because the response directly and fully addresses the question about paid leaves for full-time employees without any irrelevant information.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0005424000000000001, input_tokens=1076, output_tokens=70, verbose_logs='Statements:\n[\n    "A full-time employee receives 24 paid leaves per calendar year."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Faithfulness', threshold=0.85, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions between the actual output and the retrieval context, indicating complete alignment and faithfulness.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, eval

In [19]:
evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

evaluation_result

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              How many paid leaves does a full-time employee receive?                              │
│  │     Actual Output:      A full-time employee receives 24 paid leaves per calendar year.                      │
│  │     Expected Output:    A full-time employee receives 24 paid leaves per calendar year.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.80      │ The score is 1.00 because the response directly...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.85      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 0.75      │ The score is 0.33 because while the retrieval         │
│              │                      │       │           │ context includes the relevant statement 'Full-time    │
│              │                      │       │           │ employees receive 24 paid leaves per calendar         │
│              │                      │       │           │ year,' it also contains unrelated information such    │
│              │                      │       │           │ as 'Employees are allowed to work from home for a     │
│              │                      │       │           │ maximum of 2 days per week' and 'Employees receive    │
│              │                      │       │           │ ₹1000 per month as mobile reimbursement,' which       │
│              │                      │       │           │ reduces overall relevancy.                            │
│        PASS  │ Contextual Precision │ 1.00  │ 0.75      │ The score is 1.00 because the first node in ret...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.80      │ The score is 1.00 because the first sentence in...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How many work-from-home days are allowed per week?                                   │
│  │     Actual Output:      Employees are allowed to work from home for a maximum of 2 days per week.            │
│  │     Expected Output:    Employees can work from home for a maximum of 2 days per week.                       │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇

⚠ WARNING: No hyperparameters logged.
» ]8;id=16639310;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 164.57s | token cost: 0.0220708 USD)
» Test Results (6 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_0', success=False, metrics_data=[MetricData(name='Answer Relevancy', threshold=0.8, success=True, score=1.0, reason='The score is 1.00 because the response directly addresses the question about paid leaves for full-time employees without any irrelevant information.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost=0.0005392000000000001, input_tokens=1076, output_tokens=68, verbose_logs='Statements:\n[\n    "A full-time employee receives 24 paid leaves per calendar year."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Faithfulness', threshold=0.85, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions between the actual output and the retrieval context, indicating perfect alignment and faithfulness.', strict_mode=False, flaky=False, evaluation_model='gpt-4.1-mini', error=None, evaluation_cost

In [ ]:
# from deepeval import evaluate
# from deepeval.evaluate import AsyncConfig


# evaluation_result = evaluate(
#     test_cases=test_cases,
#     metrics=rag_metrics,

#     async_config=AsyncConfig(
#         run_async=False
#     ),
# )

# evaluation_result

## Debug one RAG test case metric-by-metric

In [20]:
debug_case = test_cases[0]

debug_metrics = [
    AnswerRelevancyMetric(
        threshold=0.80,
        model=DEEPEVAL_JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    ),
    FaithfulnessMetric(
        threshold=0.85,
        model=DEEPEVAL_JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    ),
    ContextualRelevancyMetric(
        threshold=0.75,
        model=DEEPEVAL_JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    ),
    ContextualPrecisionMetric(
        threshold=0.75,
        model=DEEPEVAL_JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    ),
    ContextualRecallMetric(
        threshold=0.80,
        model=DEEPEVAL_JUDGE_MODEL,
        include_reason=True,
        async_mode=False,
    ),
]

debug_rows = []

for metric in debug_metrics:
    metric.measure(debug_case)

    debug_rows.append({
        "metric": metric.__class__.__name__,
        "score": metric.score,
        "threshold": metric.threshold,
        "passed": metric.is_successful(),
        "reason": metric.reason,
    })

debug_df = pd.DataFrame(debug_rows)
debug_df


,metric,score,threshold,passed,reason
0,AnswerRelevancyMetric,1.000000,0.80,True,The score is 1.00 because the response directl...
1,FaithfulnessMetric,1.000000,0.85,True,The score is 1.00 because there are no contrad...
2,ContextualRelevancyMetric,0.333333,0.75,False,The score is 0.33 because while the relevant s...
3,ContextualPrecisionMetric,1.000000,0.75,True,The score is 1.00 because the first node in re...
4,ContextualRecallMetric,1.000000,0.80,True,The score is 1.00 because the sentence in the ...


In [21]:
for row in debug_rows:
    print("=" * 100)
    print("METRIC:", row["metric"])
    print("SCORE:", row["score"])
    print("PASSED:", row["passed"])
    print("REASON:")
    print(row["reason"])
    print()


METRIC: AnswerRelevancyMetric
SCORE: 1.0
PASSED: True
REASON:
The score is 1.00 because the response directly and fully answers the question about the number of paid leaves for a full-time employee without any irrelevant information.

METRIC: FaithfulnessMetric
SCORE: 1.0
PASSED: True
REASON:
The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!

METRIC: ContextualRelevancyMetric
SCORE: 0.3333333333333333
PASSED: False
REASON:
The score is 0.33 because while the relevant statement 'Full-time employees receive 24 paid leaves per calendar year.' directly answers the question, the presence of unrelated statements about work-from-home days and mobile reimbursement dilutes the overall relevancy of the retrieval context.

METRIC: ContextualPrecisionMetric
SCORE: 1.0
PASSED: True
REASON:
The score is 1.00 because the first node in retrieval contexts clearly provides the exact number of paid leaves, sta

## Full score table for every test case

In [22]:
def make_sync_metrics():
    return [
        AnswerRelevancyMetric(
            threshold=0.80,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            async_mode=False,
        ),
        FaithfulnessMetric(
            threshold=0.85,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            async_mode=False,
        ),
        ContextualRelevancyMetric(
            threshold=0.75,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            async_mode=False,
        ),
        ContextualPrecisionMetric(
            threshold=0.75,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            async_mode=False,
        ),
        ContextualRecallMetric(
            threshold=0.80,
            model=DEEPEVAL_JUDGE_MODEL,
            include_reason=True,
            async_mode=False,
        ),
    ]


In [24]:
detailed_results = []

for i, test_case in enumerate(test_cases, start=1):
    print(f"Evaluating {i}/{len(test_cases)}")

    row = {
        "input": test_case.input,
        "actual_output": test_case.actual_output,
        "expected_output": test_case.expected_output,
    }

    for metric in make_sync_metrics():
        metric.measure(test_case)

        name = metric.__class__.__name__

        row[f"{name}_score"] = metric.score
        row[f"{name}_pass"] = metric.is_successful()
        row[f"{name}_reason"] = metric.reason

    detailed_results.append(row)

detailed_results_df = pd.DataFrame(detailed_results)
detailed_results_df


Evaluating 1/6


d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Evaluating 2/6


KeyboardInterrupt: 

In [ ]:
score_columns = [
    "input",
    "AnswerRelevancyMetric_score",
    "FaithfulnessMetric_score",
    "ContextualRelevancyMetric_score",
    "ContextualPrecisionMetric_score",
    "ContextualRecallMetric_score",
]

detailed_results_df[score_columns]


NameError: name 'detailed_results_df' is not defined

In [ ]:
metric_score_columns = [
    "AnswerRelevancyMetric_score",
    "FaithfulnessMetric_score",
    "ContextualRelevancyMetric_score",
    "ContextualPrecisionMetric_score",
    "ContextualRecallMetric_score",
]

aggregate_report = (
    detailed_results_df[metric_score_columns]
    .mean()
    .to_frame("mean_score")
)

aggregate_report


In [ ]:
detailed_results_df.to_csv(
    "deepeval_rag_detailed_results.csv",
    index=False,
)

aggregate_report.to_csv(
    "deepeval_rag_aggregate_report.csv"
)

print("Saved evaluation CSV files.")


## Use with your existing RAG

Your own RAG function only needs to return:

```python
{
    "answer": "...",
    "retrieval_context": [
        "actual retrieved chunk 1",
        "actual retrieved chunk 2"
    ]
}
```

Then construct `LLMTestCase` exactly as above and run `evaluate()`.
